# Python ile Veri Bilimi ve Makine Öğrenmesi

Bu notebook, gerçekçi örneklerle veri işleme ve ML pipeline'ının tüm adımlarını gösterir.

---
## İçindekiler
1. [Veri Yükleme ve İnceleme](#1-veri-yükleme-ve-inceleme)
2. [Keşifsel Veri Analizi (EDA)](#2-keşifsel-veri-analizi-eda)
3. [Veri Temizleme ve Ön İşleme](#3-veri-temizleme-ve-ön-işleme)
4. [Özellik Mühendisliği](#4-özellik-mühendisliği)
5. [Sınıflandırma — Titanic](#5-sınıflandırma--titanic)
6. [Regresyon — Ev Fiyatı Tahmini](#6-regresyon--ev-fiyatı-tahmini)
7. [Kümeleme (K-Means)](#7-kümeleme-k-means)
8. [Model Değerlendirme ve Karşılaştırma](#8-model-değerlendirme-ve-karşılaştırma)
9. [Pipeline ile Eksiksiz Akış](#9-pipeline-ile-eksiksiz-akış)

In [ ]:
# Gerekli kütüphaneleri yükle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_diabetes, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score, silhouette_score
)
import warnings
warnings.filterwarnings('ignore')

# Tekrar üretilebilirlik için seed
np.random.seed(42)

# Grafik stili
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Tüm kütüphaneler yüklendi.")

---
## 1. Veri Yükleme ve İnceleme

İlk adım: veriyi yüklemek ve yapısını anlamak.

In [ ]:
# Iris veri seti — klasik sınıflandırma problemi
iris = load_iris(as_frame=True)
df = iris.frame
df['target_name'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Boyut:", df.shape)
print("\nİlk 5 satır:")
display(df.head())

print("\nVeri tipleri:")
print(df.dtypes)

print("\nİstatistiksel özet:")
display(df.describe().round(2))

In [ ]:
# Eksik değer kontrolü
eksik = df.isnull().sum()
print("Eksik değerler:")
print(eksik[eksik > 0] if eksik.sum() > 0 else "  ✅ Eksik değer yok")

# Hedef dağılımı
print("\nSınıf dağılımı:")
print(df['target_name'].value_counts())

---
## 2. Keşifsel Veri Analizi (EDA)

Görselleştirme ile verinin hikayesini anlıyoruz.

In [ ]:
ozellikler = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ozellik in enumerate(ozellikler):
    for sinif in df['target_name'].unique():
        veri = df[df['target_name'] == sinif][ozellik]
        axes[i].hist(veri, alpha=0.6, label=sinif, bins=15)
    axes[i].set_title(ozellik)
    axes[i].legend()
    axes[i].set_xlabel('cm')
    axes[i].set_ylabel('Frekans')

plt.suptitle('Iris Özellikleri — Sınıflara Göre Dağılım', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon ısı haritası
plt.figure(figsize=(8, 6))
korelasyon = df[ozellikler].corr()
sns.heatmap(korelasyon, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Özellikler Arası Korelasyon')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — petal uzunluk vs genişlik
renkler = {'setosa': '#e74c3c', 'versicolor': '#3498db', 'virginica': '#2ecc71'}

plt.figure(figsize=(9, 6))
for sinif, renk in renkler.items():
    alt = df[df['target_name'] == sinif]
    plt.scatter(alt['petal length (cm)'], alt['petal width (cm)'],
                c=renk, label=sinif, alpha=0.7, edgecolors='white', s=60)

plt.xlabel('Petal Uzunluğu (cm)')
plt.ylabel('Petal Genişliği (cm)')
plt.title('Iris: Petal Uzunluğu vs Genişliği')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. Veri Temizleme ve Ön İşleme

Gerçek dünyada veri nadiren temizdir. Eksik değer ve aykırı gözlem işleme.

In [ ]:
# Kasıtlı olarak kirli bir veri seti oluşturalım
np.random.seed(42)
n = 200

kirli_df = pd.DataFrame({
    'yas':     np.random.randint(18, 65, n).astype(float),
    'maas':    np.random.normal(5000, 1500, n),
    'deneyim': np.random.randint(0, 20, n).astype(float),
    'bolum':   np.random.choice(['Yazılım', 'Pazarlama', 'Finans', None], n, p=[0.4, 0.3, 0.2, 0.1]),
    'performans': np.random.choice([1, 2, 3, 4, 5], n)
})

# Eksik değer ekle
kirli_df.loc[np.random.choice(n, 20, replace=False), 'yas'] = np.nan
kirli_df.loc[np.random.choice(n, 15, replace=False), 'maas'] = np.nan

# Aykırı değer ekle
kirli_df.loc[np.random.choice(n, 5, replace=False), 'maas'] = 99999

print("Kirli veri — eksik değerler:")
print(kirli_df.isnull().sum())
print(f"\nMaaş maks: {kirli_df['maas'].max():,.0f}")

In [ ]:
temiz_df = kirli_df.copy()

# 1) Eksik sayısal değerleri medyan ile doldur
for sutun in ['yas', 'maas', 'deneyim']:
    medyan = temiz_df[sutun].median()
    eksik_sayisi = temiz_df[sutun].isnull().sum()
    temiz_df[sutun].fillna(medyan, inplace=True)
    print(f"{sutun}: {eksik_sayisi} eksik → medyan ({medyan:.1f}) ile dolduruldu")

# 2) Eksik kategorik değerleri mod ile doldur
mod = temiz_df['bolum'].mode()[0]
eks = temiz_df['bolum'].isnull().sum()
temiz_df['bolum'].fillna(mod, inplace=True)
print(f"bolum: {eks} eksik → mod ('{mod}') ile dolduruldu")

# 3) IQR yöntemiyle aykırı değerleri kırp
Q1 = temiz_df['maas'].quantile(0.25)
Q3 = temiz_df['maas'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

aykiri = ((temiz_df['maas'] < alt_sinir) | (temiz_df['maas'] > ust_sinir)).sum()
temiz_df['maas'] = temiz_df['maas'].clip(lower=alt_sinir, upper=ust_sinir)
print(f"\n{aykiri} aykırı maaş değeri [{alt_sinir:.0f} - {ust_sinir:.0f}] aralığına kırpıldı")

print(f"\n✅ Temizleme sonrası eksik: {temiz_df.isnull().sum().sum()}")

---
## 4. Özellik Mühendisliği

Ham değişkenlerden modelin öğrenebileceği anlamlı özellikler türetme.

In [ ]:
df_ozellik = temiz_df.copy()

# Yeni özellikler türet
df_ozellik['maas_yas_orani']    = df_ozellik['maas'] / df_ozellik['yas']
df_ozellik['deneyim_kare']      = df_ozellik['deneyim'] ** 2
df_ozellik['kıdemli']           = (df_ozellik['deneyim'] >= 10).astype(int)

# Bölüm → One-Hot Encoding
df_ozellik = pd.get_dummies(df_ozellik, columns=['bolum'], prefix='bolum', drop_first=True)

# Sayısal özellikleri ölçeklendir
olceklenecek = ['yas', 'maas', 'deneyim', 'maas_yas_orani', 'deneyim_kare']
scaler = StandardScaler()
df_ozellik[olceklenecek] = scaler.fit_transform(df_ozellik[olceklenecek])

print("Özellik mühendisliği sonrası sütunlar:")
print(df_ozellik.columns.tolist())
print(f"\nYeni boyut: {df_ozellik.shape}")
display(df_ozellik.head(3))

---
## 5. Sınıflandırma — Titanic

Titanic hayatta kalma tahmini: Lojistik Regresyon vs Random Forest vs Gradient Boosting

In [ ]:
# Titanic benzeri sentetik veri
np.random.seed(42)
n = 891

titanic = pd.DataFrame({
    'pclass':    np.random.choice([1, 2, 3], n, p=[0.25, 0.21, 0.54]),
    'sex':       np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'age':       np.random.normal(29, 14, n).clip(1, 80),
    'sibsp':     np.random.choice([0,1,2,3,4], n, p=[0.68,0.23,0.06,0.02,0.01]),
    'parch':     np.random.choice([0,1,2,3], n, p=[0.76,0.13,0.09,0.02]),
    'fare':      np.random.exponential(32, n).clip(5, 500),
})

# Gerçekçi hayatta kalma kuralı
hayatta_kalma_olasiligi = (
    0.2
    + 0.35 * (titanic['sex'] == 'female')
    + 0.25 * (titanic['pclass'] == 1)
    + 0.05 * (titanic['pclass'] == 2)
    - 0.15 * (titanic['age'] > 50)
).clip(0.05, 0.95)
titanic['survived'] = np.random.binomial(1, hayatta_kalma_olasiligi)

# Ön işleme
titanic['sex'] = (titanic['sex'] == 'female').astype(int)
titanic['aile_boyutu'] = titanic['sibsp'] + titanic['parch']
titanic['yalniz'] = (titanic['aile_boyutu'] == 0).astype(int)

X = titanic.drop('survived', axis=1)
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Eğitim: {X_train.shape}, Test: {X_test.shape}")
print(f"Hayatta kalma oranı: {y.mean():.2%}")

In [ ]:
# Üç model eğit ve karşılaştır
modeller = {
    'Lojistik Reg.':   LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':   RandomForestClassifier(n_estimators=100, random_state=42),
    'Grad. Boosting':  GradientBoostingClassifier(n_estimators=100, random_state=42)
}

sonuclar = {}
for isim, model in modeller.items():
    veri = X_train_s if isim == 'Lojistik Reg.' else X_train
    test_veri = X_test_s if isim == 'Lojistik Reg.' else X_test
    model.fit(veri, y_train)
    tahmin = model.predict(test_veri)
    acc = accuracy_score(y_test, tahmin)
    cv  = cross_val_score(model, veri, y_train, cv=5, scoring='accuracy').mean()
    sonuclar[isim] = {'Doğruluk': acc, 'CV Ort.': cv, 'tahmin': tahmin}
    print(f"{isim:<20} Test: {acc:.4f}  |  5-fold CV: {cv:.4f}")

In [ ]:
# En iyi model için Confusion Matrix
en_iyi = max(sonuclar, key=lambda k: sonuclar[k]['Doğruluk'])
print(f"En iyi model: {en_iyi}\n")

cm = confusion_matrix(y_test, sonuclar[en_iyi]['tahmin'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Hayatta Kalmadı', 'Hayatta Kaldı'],
            yticklabels=['Hayatta Kalmadı', 'Hayatta Kaldı'])
plt.title(f'Confusion Matrix — {en_iyi}')
plt.ylabel('Gerçek')
plt.xlabel('Tahmin')
plt.tight_layout()
plt.show()

print(classification_report(y_test, sonuclar[en_iyi]['tahmin'],
                             target_names=['Hayatta Kalmadı', 'Hayatta Kaldı']))

---
## 6. Regresyon — Ev Fiyatı Tahmini

Doğrusal Regresyon, Ridge ve özellik önemi analizi.

In [ ]:
# Diyabet veri seti — regresyon
diabet = load_diabetes(as_frame=True)
X_d = diabet.data
y_d = diabet.target

X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

reg_modeller = {
    'Doğrusal Reg.': LinearRegression(),
    'Ridge (α=1)':   Ridge(alpha=1.0),
    'Ridge (α=10)':  Ridge(alpha=10.0)
}

print(f"{'Model':<20} {'R²':>8} {'RMSE':>10}")
print('-' * 40)
reg_sonuclar = {}
for isim, model in reg_modeller.items():
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    reg_sonuclar[isim] = (y_pred, r2, rmse)
    print(f"{isim:<20} {r2:>8.4f} {rmse:>10.2f}")

In [ ]:
# Gerçek vs Tahmin grafiği
y_pred_lr, r2_lr, _ = reg_sonuclar['Doğrusal Reg.']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gerçek vs Tahmin
axes[0].scatter(y_te, y_pred_lr, alpha=0.6, edgecolors='white', s=50)
lim = [min(y_te.min(), y_pred_lr.min()), max(y_te.max(), y_pred_lr.max())]
axes[0].plot(lim, lim, 'r--', lw=2, label='Mükemmel tahmin')
axes[0].set_xlabel('Gerçek')
axes[0].set_ylabel('Tahmin')
axes[0].set_title(f'Gerçek vs Tahmin (R²={r2_lr:.3f})')
axes[0].legend()

# Kalıntı (Residual) dağılımı
kalinti = y_te - y_pred_lr
axes[1].hist(kalinti, bins=25, edgecolor='white', color='#3498db', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Kalıntı (Gerçek − Tahmin)')
axes[1].set_ylabel('Frekans')
axes[1].set_title('Kalıntı Dağılımı')

plt.tight_layout()
plt.show()

---
## 7. Kümeleme (K-Means)

Gözetimsiz öğrenmede veri noktalarını gruplandırma.

In [ ]:
# Sentetik 3 kümeli veri
X_k, y_gercek = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Optimum k bul — Elbow yöntemi
ataletler = []
k_aralik = range(1, 11)
for k in k_aralik:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_k)
    ataletler.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_aralik, ataletler, 'bo-', linewidth=2, markersize=8)
plt.axvline(3, color='red', linestyle='--', label='Optimum k=3')
plt.xlabel('Küme Sayısı (k)')
plt.ylabel('Atalet (İnertia)')
plt.title('Elbow Yöntemi ile Optimum k Seçimi')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# k=3 ile K-Means uygula
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kume_etiketleri = kmeans.fit_predict(X_k)
silhouette = silhouette_score(X_k, kume_etiketleri)

print(f"Silhouette Skoru: {silhouette:.4f}  (1'e yakın → iyi ayrışım)")

renkler = ['#e74c3c', '#3498db', '#2ecc71']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sinif in range(3):
    mask = kume_etiketleri == sinif
    axes[0].scatter(X_k[mask, 0], X_k[mask, 1], c=renkler[sinif],
                    label=f'Küme {sinif+1}', alpha=0.7, s=50)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                marker='X', s=200, c='black', zorder=5, label='Merkez')
axes[0].set_title('K-Means Sonucu')
axes[0].legend()

for sinif in range(3):
    mask = y_gercek == sinif
    axes[1].scatter(X_k[mask, 0], X_k[mask, 1], c=renkler[sinif],
                    label=f'Gerçek {sinif+1}', alpha=0.7, s=50)
axes[1].set_title('Gerçek Kümeler')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. Model Değerlendirme ve Karşılaştırma

GridSearchCV ile hiperparametre optimizasyonu ve özellik önemi.

In [ ]:
# Iris ile Random Forest — GridSearch
X_iris = iris.data
y_iris = iris.target

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 3, 5],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_tr_i, y_tr_i)

print(f"En iyi parametreler : {grid_search.best_params_}")
print(f"En iyi CV skoru     : {grid_search.best_score_:.4f}")
print(f"Test doğruluğu      : {grid_search.score(X_te_i, y_te_i):.4f}")

In [ ]:
# Özellik önemi
en_iyi_model = grid_search.best_estimator_
onem = en_iyi_model.feature_importances_
ozellik_isimleri = iris.feature_names

sirali_idx = np.argsort(onem)[::-1]

plt.figure(figsize=(8, 4))
renkler_bar = ['#e74c3c' if o > 0.3 else '#3498db' for o in onem[sirali_idx]]
plt.bar(range(len(onem)), onem[sirali_idx], color=renkler_bar, edgecolor='white')
plt.xticks(range(len(onem)), [ozellik_isimleri[i] for i in sirali_idx], rotation=15)
plt.ylabel('Önem Skoru')
plt.title('Random Forest — Özellik Önemi')
plt.tight_layout()
plt.show()

---
## 9. Pipeline ile Eksiksiz Akış

Gerçek bir ML projesinde ön işleme + model birlikte `Pipeline` içinde paketlenir.  
Bu, veri sızıntısını önler ve kodu temiz tutar.

In [ ]:
# Karma tipli özellikler için tam pipeline
np.random.seed(42)
n = 500

musteri = pd.DataFrame({
    'yas':      np.random.normal(40, 12, n),
    'gelir':    np.random.normal(50000, 20000, n),
    'kredi_sk': np.random.randint(300, 850, n).astype(float),
    'meslek':   np.random.choice(['Mühendis', 'Öğretmen', 'Doktor', 'Serbest'], n),
    'ev_sahibi': np.random.choice(['Evet', 'Hayır'], n),
})

# Eksik değer ekle
musteri.loc[np.random.choice(n, 40, replace=False), 'kredi_sk'] = np.nan
musteri.loc[np.random.choice(n, 25, replace=False), 'meslek'] = np.nan

# Hedef: kredi onayı
musteri['onay'] = ((musteri['kredi_sk'].fillna(500) > 600) &
                   (musteri['gelir'] > 40000)).astype(int)

X = musteri.drop('onay', axis=1)
y = musteri['onay']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Sayısal ve kategorik sütunları ayır
sayisal = ['yas', 'gelir', 'kredi_sk']
kategorik = ['meslek', 'ev_sahibi']

# Ön işleme adımları
sayisal_islemci = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

kategorik_islemci = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

on_isleme = ColumnTransformer([
    ('sayisal',   sayisal_islemci,   sayisal),
    ('kategorik', kategorik_islemci, kategorik)
])

# Tam pipeline
pipeline = Pipeline([
    ('on_isleme', on_isleme),
    ('model',     RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_tr, y_tr)
acc = pipeline.score(X_te, y_te)
cv  = cross_val_score(pipeline, X_tr, y_tr, cv=5).mean()

print(f"Test Doğruluğu : {acc:.4f}")
print(f"5-fold CV Ort. : {cv:.4f}")
print(f"\n{'='*50}")
print(classification_report(y_te, pipeline.predict(X_te),
                             target_names=['Reddedildi', 'Onaylandı']))

In [ ]:
# Yeni müşteriler için tahmin
yeni_musteriler = pd.DataFrame({
    'yas':       [28, 55, 35],
    'gelir':     [35000, 80000, 52000],
    'kredi_sk':  [580, 750, np.nan],
    'meslek':    ['Mühendis', 'Doktor', None],
    'ev_sahibi': ['Hayır', 'Evet', 'Evet']
})

tahminler = pipeline.predict(yeni_musteriler)
olasiliklar = pipeline.predict_proba(yeni_musteriler)[:, 1]

print("Yeni Müşteri Tahminleri:")
print(f"{'Müşteri':<12} {'Karar':<15} {'Onay Olasılığı':>16}")
print('-' * 45)
for i, (t, p) in enumerate(zip(tahminler, olasiliklar)):
    karar = '✅ Onaylandı' if t == 1 else '❌ Reddedildi'
    print(f"Müşteri {i+1:<5} {karar:<15} {p:>14.1%}")

---
## Özet

| Adım | Araçlar |
|---|---|
| Veri Yükleme | `pandas`, `sklearn.datasets` |
| EDA | `matplotlib`, `seaborn` |
| Temizleme | `fillna`, `clip`, IQR |
| Özellik Müh. | `get_dummies`, `StandardScaler` |
| Sınıflandırma | `LogisticRegression`, `RandomForest`, `GradientBoosting` |
| Regresyon | `LinearRegression`, `Ridge` |
| Kümeleme | `KMeans`, `silhouette_score` |
| Optimizasyon | `GridSearchCV`, `cross_val_score` |
| Pipeline | `Pipeline`, `ColumnTransformer` |

> **Sıradaki adım:** XGBoost/LightGBM, nöral ağlar, zaman serisi analizi, model açıklanabilirliği (SHAP).